# Lunar Router SDK Examples

Unified LLM interface for **13 providers** — like LiteLLM / OpenRouter.

```
pip install -e ".[openai,anthropic,api]"
```

## 1. Quick Start — One-liner Completion

In [9]:
import lunar_router as lr

# Call any model with provider/model syntax
response = lr.completion(
    model="openai/gpt-4o-mini",
    messages=[{"role": "user", "content": "What is 2+2? Reply in one word."}],
    temperature=0,
)

print(response.choices[0].message.content)
print(f"Model:    {response.model}")
print(f"Tokens:   {response.usage.prompt_tokens} in / {response.usage.completion_tokens} out")
print(f"Cost:     ${response._cost:.6f}")
print(f"Latency:  {response._latency_ms:.0f}ms")
print(f"Provider: {response._provider}")

Four.
Model:    gpt-4o-mini-2024-07-18
Tokens:   19 in / 2 out
Cost:     $0.000004
Latency:  2498ms
Provider: openai


## 2. Multiple Providers — Same Interface

In [ ]:
# Every provider uses the exact same API
models = [
    "openai/gpt-4o-mini",
    "anthropic/claude-haiku-4-5-20251001",
    "mistral/mistral-small-latest",
    "groq/llama-3.3-70b-versatile",
    "deepseek/deepseek-chat",
    "gemini/gemini-2.0-flash",
]

prompt = [{"role": "user", "content": "Name one planet. Reply with just the name."}]

for model in models:
    try:
        resp = lr.completion(model=model, messages=prompt, temperature=0, max_tokens=10)
        text = resp.choices[0].message.content.strip()
        print(f"{model:45s} → {text:15s}  (${resp._cost:.6f}, {resp._latency_ms:.0f}ms)")
    except Exception as e:
        print(f"{model:45s} → ERROR: {e}")

## 3. Streaming

In [ ]:
# Streaming works with all 13 providers (Anthropic & Bedrock SSE auto-translated)
print("Streaming from OpenAI: ", end="", flush=True)
for chunk in lr.completion(
    model="openai/gpt-4o-mini",
    messages=[{"role": "user", "content": "Count from 1 to 10, comma separated."}],
    stream=True,
):
    delta = chunk.choices[0].delta
    if delta and delta.content:
        print(delta.content, end="", flush=True)
print()

In [ ]:
# Streaming with Anthropic (SSE format translated automatically by Go engine)
print("Streaming from Anthropic: ", end="", flush=True)
for chunk in lr.completion(
    model="anthropic/claude-haiku-4-5-20251001",
    messages=[{"role": "user", "content": "Count from 1 to 10, comma separated."}],
    stream=True,
):
    delta = chunk.choices[0].delta
    if delta and delta.content:
        print(delta.content, end="", flush=True)
print()

## 4. Auto-routing (model="auto")

When using the Go engine, `model="auto"` activates semantic routing — the engine
embeds the prompt and picks the best model based on trained cluster profiles.

In [ ]:
# The Go engine must be running with weights loaded for auto-routing
try:
    resp = lr.completion(
        model="auto",
        messages=[{"role": "user", "content": "Write a Python quicksort implementation."}],
        force_engine=True,
    )
    print(f"Router selected: {resp.model}")
    print(f"Routing metadata: {resp._routing}")
    print(resp.choices[0].message.content[:200])
except Exception as e:
    print(f"Auto-routing requires the Go engine with weights: {e}")

## 5. Fallbacks & Retries

In [ ]:
# If the primary model fails, fallbacks are tried in order
resp = lr.completion(
    model="openai/gpt-4o-mini",
    messages=[{"role": "user", "content": "Hello!"}],
    fallbacks=[
        "anthropic/claude-haiku-4-5-20251001",
        "groq/llama-3.3-70b-versatile",
    ],
    num_retries=1,  # retry each model once before moving to next
)
print(f"Response from: {resp.model}")
print(resp.choices[0].message.content)

## 6. Router Class — Load Balancing & Failover

Map logical model names to multiple provider deployments with
automatic load balancing and fallback chains.

In [ ]:
router = lr.Router(
    model_list=[
        # Two deployments behind the "smart" alias
        {"model_name": "smart", "model": "openai/gpt-4o-mini"},
        {"model_name": "smart", "model": "anthropic/claude-haiku-4-5-20251001"},
        # Single deployment for "fast"
        {"model_name": "fast", "model": "groq/llama-3.3-70b-versatile"},
    ],
    fallbacks=[{"smart": ["deepseek/deepseek-chat"]}],
    strategy="round-robin",  # or: "least-cost", "lowest-latency", "weighted-random"
    num_retries=1,
)

# Each call round-robins between OpenAI and Anthropic
for i in range(4):
    resp = router.completion(
        model="smart",
        messages=[{"role": "user", "content": f"Say 'hello {i}' and nothing else."}],
        temperature=0,
    )
    print(f"Call {i}: {resp.model:30s} → {resp.choices[0].message.content.strip()}")

In [ ]:
# Check per-deployment stats
import json
print(json.dumps(router.get_stats(), indent=2))

## 7. Cost Tracking

In [ ]:
from lunar_router import model_cost, get_model_info, supported_models

# Check pricing for a model
info = get_model_info("gpt-4o-mini")
print("gpt-4o-mini pricing:")
print(f"  Input:  ${info['input_cost_per_token'] * 1_000_000:.2f} / 1M tokens")
print(f"  Output: ${info['output_cost_per_token'] * 1_000_000:.2f} / 1M tokens")
print(f"  Context: {info['max_input_tokens']:,} tokens")
print()

# Calculate cost for a hypothetical call
cost = model_cost("gpt-4o-mini", input_tokens=1000, output_tokens=500)
print(f"Cost for 1K in + 500 out: ${cost:.6f}")
print()

# Total models with pricing data
print(f"Models with pricing: {len(supported_models())}")

In [ ]:
# Compare costs across providers for the same task
models_to_compare = [
    "gpt-4o-mini",
    "claude-haiku-4-5-20251001",
    "mistral-small-latest",
    "llama-3.3-70b-versatile",
    "deepseek-chat",
    "gemini-2.0-flash",
]

print(f"{'Model':<35} {'1K in + 1K out':>15} {'10K in + 2K out':>15}")
print("-" * 67)
for m in models_to_compare:
    c1 = model_cost(m, 1000, 1000)
    c2 = model_cost(m, 10000, 2000)
    print(f"{m:<35} ${c1:>13.6f} ${c2:>13.6f}")

## 8. Model Name Parsing

In [ ]:
from lunar_router import parse_model

# Explicit provider/model format
print(parse_model("openai/gpt-4o-mini"))                    # ('openai', 'gpt-4o-mini')
print(parse_model("anthropic/claude-haiku-4-5-20251001"))    # ('anthropic', 'claude-...')
print(parse_model("groq/llama-3.3-70b-versatile"))           # ('groq', 'llama-...')
print()

# Auto-detection from model name prefix
print(parse_model("gpt-4o-mini"))              # ('openai', 'gpt-4o-mini')
print(parse_model("claude-sonnet-4-20250514")) # ('anthropic', 'claude-...')
print(parse_model("gemini-2.0-flash"))         # ('gemini', 'gemini-...')
print(parse_model("deepseek-chat"))            # ('deepseek', 'deepseek-chat')
print(parse_model("command-r-plus"))           # ('cohere', 'command-r-plus')

## 9. Drop-in OpenAI SDK Replacement

Point the standard OpenAI SDK at the Go engine. All 13 providers
are available through the same endpoint.

In [ ]:
from openai import OpenAI

# Point at the Go engine (default: localhost:8080)
client = OpenAI(
    base_url="http://localhost:8080/v1",
    api_key="any",  # engine handles auth per-provider
)

# Use provider/model syntax — engine routes to the right provider
resp = client.chat.completions.create(
    model="openai/gpt-4o-mini",
    messages=[{"role": "user", "content": "Hello from OpenAI SDK!"}],
    max_tokens=50,
)
print(resp.choices[0].message.content)

In [ ]:
# Same client, different provider — no code changes needed
resp = client.chat.completions.create(
    model="mistral/mistral-small-latest",
    messages=[{"role": "user", "content": "Hello from OpenAI SDK via Mistral!"}],
    max_tokens=50,
)
print(resp.choices[0].message.content)

In [ ]:
# Streaming via OpenAI SDK
stream = client.chat.completions.create(
    model="openai/gpt-4o-mini",
    messages=[{"role": "user", "content": "Count from 1 to 5."}],
    stream=True,
)
for chunk in stream:
    if chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)
print()

## 10. Async Completion

In [ ]:
import asyncio

async def run_parallel():
    """Call multiple providers concurrently."""
    tasks = [
        lr.acompletion(
            model=m,
            messages=[{"role": "user", "content": "Say your model name."}],
            max_tokens=30,
        )
        for m in ["openai/gpt-4o-mini", "mistral/mistral-small-latest", "groq/llama-3.3-70b-versatile"]
    ]
    results = await asyncio.gather(*tasks, return_exceptions=True)
    for r in results:
        if isinstance(r, Exception):
            print(f"ERROR: {r}")
        else:
            print(f"{r.model:30s} → {r.choices[0].message.content.strip()[:60]}")

await run_parallel()

## 11. Tool Calling

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get current weather for a city",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "City name"},
                },
                "required": ["city"],
            },
        },
    }
]

resp = lr.completion(
    model="openai/gpt-4o-mini",
    messages=[{"role": "user", "content": "What's the weather in Tokyo?"}],
    tools=tools,
    tool_choice="auto",
)

msg = resp.choices[0].message
if msg.tool_calls:
    for tc in msg.tool_calls:
        print(f"Tool call: {tc.function.name}({tc.function.arguments})")
else:
    print(msg.content)

## 12. Router Strategies Compared

In [ ]:
# Least-cost strategy picks the cheapest provider first
cost_router = lr.Router(
    model_list=[
        {"model_name": "chat", "model": "openai/gpt-4o-mini"},
        {"model_name": "chat", "model": "deepseek/deepseek-chat"},
        {"model_name": "chat", "model": "groq/llama-3.3-70b-versatile"},
    ],
    strategy="least-cost",
)

resp = cost_router.completion(
    model="chat",
    messages=[{"role": "user", "content": "Hello!"}],
    max_tokens=10,
)
print(f"Least-cost picked: {resp.model} (cost: ${resp._cost:.6f})")

In [ ]:
# Weighted-random distributes traffic by weight
weighted_router = lr.Router(
    model_list=[
        {"model_name": "chat", "model": "openai/gpt-4o-mini", "weight": 3},
        {"model_name": "chat", "model": "mistral/mistral-small-latest", "weight": 1},
    ],
    strategy="weighted-random",
)

counts = {}
for _ in range(20):
    resp = weighted_router.completion(
        model="chat",
        messages=[{"role": "user", "content": "Hi"}],
        max_tokens=5,
    )
    counts[resp.model] = counts.get(resp.model, 0) + 1

print("Traffic distribution (expected ~75/25):")
for model, count in sorted(counts.items()):
    print(f"  {model}: {count}/20 ({count/20*100:.0f}%)")

## 13. Direct vs Engine Mode

In [ ]:
# force_engine=True: always proxy through Go engine (gets cost tracking, analytics)
resp = lr.completion(
    model="openai/gpt-4o-mini",
    messages=[{"role": "user", "content": "Hello via engine!"}],
    force_engine=True,
    max_tokens=20,
)
print(f"Via engine: {resp.choices[0].message.content}")

# force_direct=True: call provider API directly (skip engine)
resp = lr.completion(
    model="openai/gpt-4o-mini",
    messages=[{"role": "user", "content": "Hello direct!"}],
    force_direct=True,
    max_tokens=20,
)
print(f"Direct:     {resp.choices[0].message.content}")

## 14. All 13 Supported Providers

| Provider | Model Syntax | Env Var |
|----------|-------------|----------|
| OpenAI | `openai/gpt-4o-mini` | `OPENAI_API_KEY` |
| Anthropic | `anthropic/claude-haiku-4-5-20251001` | `ANTHROPIC_API_KEY` |
| Gemini | `gemini/gemini-2.0-flash` | `GEMINI_API_KEY` |
| Mistral | `mistral/mistral-small-latest` | `MISTRAL_API_KEY` |
| Groq | `groq/llama-3.3-70b-versatile` | `GROQ_API_KEY` |
| DeepSeek | `deepseek/deepseek-chat` | `DEEPSEEK_API_KEY` |
| Perplexity | `perplexity/sonar` | `PERPLEXITY_API_KEY` |
| Cerebras | `cerebras/llama3.1-70b` | `CEREBRAS_API_KEY` |
| SambaNova | `sambanova/Meta-Llama-3.1-70B-Instruct` | `SAMBANOVA_API_KEY` |
| Together | `together/meta-llama/Llama-3.3-70B-Instruct-Turbo` | `TOGETHER_API_KEY` |
| Fireworks | `fireworks/accounts/fireworks/models/llama-v3p1-70b-instruct` | `FIREWORKS_API_KEY` |
| Cohere | `cohere/command-r-plus` | `COHERE_API_KEY` |
| AWS Bedrock | `bedrock/amazon.titan-text-express-v1` | `AWS_ACCESS_KEY_ID` + `AWS_SECRET_ACCESS_KEY` |

In [ ]:
# Check which providers are configured (have API keys)
import os
from lunar_router.sdk import PROVIDERS

print(f"{'Provider':<15} {'Env Var':<30} {'Configured'}")
print("-" * 60)
for name, cfg in sorted(PROVIDERS.items()):
    env_var = cfg.get("api_key_env", "")
    configured = "YES" if os.environ.get(env_var) else "no"
    print(f"{name:<15} {env_var:<30} {configured}")